# Consumer Shopping Trends: Data Cleaning

**Author:** Namit

**Dataset:** `customer_shopping_behavior.csv`: 3,900 rows, 18 columns of retail transaction data (customer demographics, purchase details, and behavior fields)

## Purpose
This notebook takes the raw shopping trends dataset and prepares it for downstream analysis (EDA in SQL Server). The focus here is entirely on cleaning and structuring the data correctly, not on drawing conclusions yet. Specifically, this notebook:

1. Profiles the raw data (structure, types, missing values)
2. Imputes missing `Review Rating` values in a way that respects category-level differences
3. Standardizes column names to a consistent `snake_case` format
4. Engineers two new features (`age_group`, `purchase_frequency_days`) to make later grouping and segmentation easier
5. Identifies and removes a fully redundant column
6. Exports the cleaned dataset for use in SQL

## Output
The cleaned file (`cleaned_customer_trend.csv`) is exported at the end and used as the source table for EDA work in SQL Server (see the companion SQL analysis in this repo).

## 1. Load the raw data

In [1]:
# Load the raw dataset into a DataFrame for inspection and cleaning using pandas

import pandas as pd
df = pd.read_csv("customer_shopping_behavior.csv")

### Initial look at the data
A quick preview to confirm the file loaded correctly and to get a first sense of the columns and value formats before doing any structural checks.

In [2]:
df.head()

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


## 2. Check structure and data types

`df.info()` gives row count, column count, and non-null counts per column in one view, the fastest way to spot missing data and confirm each column's dtype is sensible before doing any transformations.

**Result:** 3,900 rows across 18 columns. All columns are fully populated except `Review Rating`, which has 3,863 non-null values (37 missing).

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Gender                  3900 non-null   object 
 3   Item Purchased          3900 non-null   object 
 4   Category                3900 non-null   object 
 5   Purchase Amount (USD)   3900 non-null   int64  
 6   Location                3900 non-null   object 
 7   Size                    3900 non-null   object 
 8   Color                   3900 non-null   object 
 9   Season                  3900 non-null   object 
 10  Review Rating           3863 non-null   float64
 11  Subscription Status     3900 non-null   object 
 12  Shipping Type           3900 non-null   object 
 13  Discount Applied        3900 non-null   object 
 14  Promo Code Used         3900 non-null   

## 3. Summary statistics

`df.describe(include='all')` covers both numeric and categorical columns in one call, giving distribution stats (mean, std, quartiles) for numeric fields and frequency counts (most common value, its count) for categorical fields. This is a fast way to sanity-check ranges (e.g., `Age` running 18–70, `Purchase Amount` running $20–$100) and spot any categorical column with an unexpectedly dominant or unbalanced value before deciding how to handle it.

In [4]:
df.describe(include='all')

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
count,3900.000000,3900.000000,3900,3900,3900,3900.000000,3900,3900,3900,3900,3863.000000,3900,3900,3900,3900,3900.000000,3900,3900
unique,NaN,NaN,2,25,4,NaN,50,4,25,4,NaN,2,6,2,2,NaN,6,7
top,NaN,NaN,Male,Blouse,Clothing,NaN,Montana,M,Olive,Spring,NaN,No,Free Shipping,No,No,NaN,PayPal,Every 3 Months
freq,NaN,NaN,2652,171,1737,NaN,96,1755,177,999,NaN,2847,675,2223,2223,NaN,677,584
mean,1950.500000,44.068462,NaN,NaN,NaN,59.764359,NaN,NaN,NaN,NaN,3.750065,NaN,NaN,NaN,NaN,25.351538,NaN,NaN
std,1125.977353,15.207589,NaN,NaN,NaN,23.685392,NaN,NaN,NaN,NaN,0.716983,NaN,NaN,NaN,NaN,14.447125,NaN,NaN
min,1.000000,18.000000,NaN,NaN,NaN,20.000000,NaN,NaN,NaN,NaN,2.500000,NaN,NaN,NaN,NaN,1.000000,NaN,NaN
25%,975.750000,31.000000,NaN,NaN,NaN,39.000000,NaN,NaN,NaN,NaN,3.100000,NaN,NaN,NaN,NaN,13.000000,NaN,NaN
50%,1950.500000,44.000000,NaN,NaN,NaN,60.000000,NaN,NaN,NaN,NaN,3.800000,NaN,NaN,NaN,NaN,25.000000,NaN,NaN
75%,2925.250000,57.000000,NaN,NaN,NaN,81.000000,NaN,NaN,NaN,NaN,4.400000,NaN,NaN,NaN,NaN,38.000000,NaN,NaN


## 4. Confirm which columns have missing values

Isolating the null count per column confirms what `.info()` suggested: **`Review Rating` is the only column with missing data, 37 out of 3,900 rows (~0.9%)**. Every other column is fully populated, so the cleaning effort can focus entirely on this one field.

In [5]:
df.isnull().sum()

,0
Customer ID,0
Age,0
Gender,0
Item Purchased,0
Category,0
Purchase Amount (USD),0
Location,0
Size,0
Color,0
Season,0


## 5. Impute missing `Review Rating` values

Rather than filling missing ratings with a single global median (which would ignore the fact that different product categories tend to be rated differently), I impute each missing value using the **median rating within that row's own `Category`**. This keeps the imputed values consistent with how that category is actually rated elsewhere in the data, instead of pulling every missing value toward one overall average.

`groupby('Category')['Review Rating'].transform(...)` applies this per-group median while preserving the original row order and index — no rows are dropped or reordered.

In [6]:
# Fill missing Review Rating values using the median rating for that row's Category,
# rather than a single global median, so imputed values stay consistent with
# how that specific category tends to be rated.

df['Review Rating'] = df.groupby('Category')['Review Rating'].transform(lambda x: x.fillna(x.median()))

**Verification:** re-running the null check confirms all 37 missing `Review Rating` values have been filled, every column now shows 0 nulls.

In [7]:
df.isnull().sum()

,0
Customer ID,0
Age,0
Gender,0
Item Purchased,0
Category,0
Purchase Amount (USD),0
Location,0
Size,0
Color,0
Season,0


## 6. Standardize column names

The raw column headers are inconsistent for programmatic use — mixed case, spaces, and parentheses (e.g., `Purchase Amount (USD)`). I normalize all column names to lowercase `snake_case` (e.g., `purchase_amount_usd`) so they're safe to reference directly in both pandas and later in SQL without needing to quote or bracket identifiers.

In [8]:
# Normalize all column names to lowercase snake_case: lowercase, spaces to underscores,
# and strip parentheses. Makes columns safe to reference directly in pandas and SQL
# without quoting.

df.columns = df.columns.str.lower()
df.columns = df.columns.str.replace(' ', '_')
df.columns = df.columns.str.replace('(', '')
df.columns = df.columns.str.replace(')', '')
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount_usd', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'promo_code_used', 'previous_purchases',
       'payment_method', 'frequency_of_purchases'],
      dtype='object')

## 7. Feature engineering: `age_group`

I bucket the continuous `age` column into four labeled groups (`young_adult`, `adult`, `middle_aged`, `senior`) using `pd.cut`. This trades some granularity for interpretability, it's much easier to compare purchase behavior across four labeled cohorts in EDA than to eyeball trends across 53 distinct ages.

Bin edges (18–25, 26–40, 41–60, 61–100) are a reasonable general-purpose age segmentation for consumer behavior analysis, chosen to roughly separate early-career, established, midlife, and older shoppers.

In [9]:
# create a column age_group

labels = ['young_adult', 'adult', 'middle_aged', 'senior']
df['age_group'] = pd.cut(df['age'], bins=[18, 25, 40, 60, 100], labels=labels, include_lowest=True)
df[['age', 'age_group']].head(10)

,age,age_group
0,55,middle_aged
1,19,young_adult
2,50,middle_aged
3,21,young_adult
4,45,middle_aged
5,46,middle_aged
6,63,senior
7,27,adult
8,26,adult
9,57,middle_aged


## 8. Feature engineering: `purchase_frequency_days`

The original `frequency_of_purchases` column is a categorical label (`Weekly`, `Monthly`, `Annually`, etc.), which is descriptive but not directly usable for numeric comparisons (e.g., ranking customers by how often they actually buy). I map each category to an approximate number of days between purchases, giving a continuous field that can be aggregated, averaged, or correlated with other numeric columns in SQL.

In [10]:
# create column purchase_frequency_days

frequency_mapping ={
    'Fortnightly': 14,
    'Weekly': 7,
    'Bi-Weekly': 14,
    'Monthly': 30,
    'Quarterly': 90,
    'Annually': 365,
    'Every 3 Months': 90
}
df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_mapping)
df[['frequency_of_purchases', 'purchase_frequency_days']].head(10)

,frequency_of_purchases,purchase_frequency_days
0,Fortnightly,14
1,Fortnightly,14
2,Weekly,7
3,Weekly,7
4,Annually,365
5,Weekly,7
6,Quarterly,90
7,Weekly,7
8,Annually,365
9,Quarterly,90


## 9. Check for redundant columns

`discount_applied` and `promo_code_used` look like they might be tracking the same underlying event under two different names. Before assuming that, I check a sample of both columns side by side.

In [11]:
df[['discount_applied', 'promo_code_used']].head(10)

,discount_applied,promo_code_used
0,Yes,Yes
1,Yes,Yes
2,Yes,Yes
3,Yes,Yes
4,Yes,Yes
5,Yes,Yes
6,Yes,Yes
7,Yes,Yes
8,Yes,Yes
9,Yes,Yes


Comparing the two columns across **all 3,900 rows** (not just the sample above) confirms they are identical in every row, `discount_applied` and `promo_code_used` always match. This means `promo_code_used` carries no information beyond what `discount_applied` already captures, so it's safe to drop.

In [12]:
(df['discount_applied'] == df['promo_code_used']).all()

np.True_

## 10. Drop the redundant column

In [13]:
# discount_applied and promo_code_used were confirmed identical across every row above,
# so promo_code_used is redundant and safe to drop.

df = df.drop('promo_code_used', axis=1)
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount_usd', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'previous_purchases', 'payment_method',
       'frequency_of_purchases', 'age_group', 'purchase_frequency_days'],
      dtype='object')

## 11. Export the cleaned dataset

The cleaned DataFrame is written to `cleaned_customer_trend.csv` and downloaded locally. This file is the single source of truth used for the SQL EDA phase, it now has: no missing values, standardized snake_case column names, two new engineered features (`age_group`, `purchase_frequency_days`), and one redundant column removed (down from 18 to 19 columns net, replacing 1 dropped with 2 added).

In [14]:
# Export the cleaned dataset — this becomes the source file imported into SQL Server
# for the EDA phase of this project.

df.to_csv("cleaned_customer_trend.csv", index=False)

from google.colab import files
files.download("cleaned_customer_trend.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Summary

| Step | Change |
|---|---|
| Missing values | 37 missing `review_rating` values imputed using category-level median |
| Column naming | All 18 original columns renamed to lowercase snake_case |
| New features | `age_group` (4-bucket age segmentation), `purchase_frequency_days` (numeric purchase cadence) |
| Redundant data | `promo_code_used` dropped — confirmed identical to `discount_applied` across all 3,900 rows |
| Final shape | 3,900 rows × 19 columns, zero missing values |

**Next step:** this cleaned file is imported into SQL Server as the source table for exploratory data analysis, covering distributions, category-level breakdowns, and purchase behavior patterns in T-SQL.